### PromptTemplate 
* [PromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html#langchain_core.prompts.prompt.PromptTemplate)
* [ChatPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
* [ChatMessagePromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatMessagePromptTemplate.html#langchain_core.prompts.chat.ChatMessagePromptTemplate)
* [FewShotPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.few_shot.FewShotPromptTemplate.html#langchain_core.prompts.few_shot.FewShotPromptTemplate)
* PartialPrompt

In [ ]:
# uv add python-dotenv langchain langchain-openai

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:5])

gsk_D


##### 1) PromptTemplate 의 from_template() 함수 사용
* 주로 LLM(텍스트 완성형 모델, ex. Ollama, GPT-3.5)과 함께 사용
* 하나의 문자열 프롬프트를 생성

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

# llm = ChatOpenAI(model="gpt-3.5-turbo-0125")
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    #model="meta-llama/llama-4-scout-17b-16e-instruct",  # Spring AI와 동일한 모델
    model="openai/gpt-oss-120b",
    #model="moonshotai/kimi-k2-instruct-0905",
    temperature=0.7
)

chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"model_name":"ChatGPT", "count":3})
pprint(response)

('ChatGPT는 인터넷에서 모은 방대한 텍스트를 단어 순서 맞추기 문제처럼 바꿔 놓고, 이를 반복해서 풀면서 스스로 통계적 패턴을 '
 '기억합니다.  \n'
 '학습이 끝난 모델은 주어진 앞 문장을 바탕으로 다음에 올 법한 단어의 확률을 계산해 한 글자씩 이어가며 답변을 만듭니다.  \n'
 '이 과정에서 사람의 지식은 들어가지 않고, 오직 문자 사이의 관계만 학습되므로 틀린 정보를 당당히 내놓을 수도 있습니다.')


##### 2) PromptTemplate 결합하기
* 동일한 Prompt 패턴을 사용하지만 여러 개의 질문을 작성해서 LLM을 실행할 수도 있습니다.

In [5]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

# 템플릿에 값을 채워서 프롬프트를 완성
#filled_prompt = prompt_template.format(model_name="ChatGPT", count=3)

# 문자열 템플릿 결합 (PromptTemplate + PromptTemplate + 문자열)
combined_prompt = (
              prompt_template
              + PromptTemplate.from_template("\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요")
              + "\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요."
)
combined_prompt.format(model_name="ChatGPT", count=3, language="한국어")
print(combined_prompt)

#llm = ChatOpenAI(model="gpt-3.5-turbo-0125")
chain = combined_prompt | llm | StrOutputParser()
response = chain.invoke({"model_name":"Gemini", "count":3, "language":"한국어"})

pprint(response)

input_variables=['count', 'language', 'model_name'] input_types={} partial_variables={} template='{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요.\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요.'
('Gemini 모델은 대규모 텍스트·이미지·오디오 등 다중 모달 데이터를 동시에 학습해 패턴을 포착하고, 인코더-디코더 기반 트랜스포머 '
 '구조로 입력을 이해·생성하며, 강화학습과 인간 피드백(RLHF)로 정답률과 안전성을 반복해 끌어올린다.\n'
 '\n'
 '장점 요약  \n'
 '- 텍스트뿐 아니라 이미지·동영상·음성까지 한 번에 이해·생성하는 ‘진정한 다중 모달’  \n'
 '- 학습 마지막 단계에서 사고력·추론 문제를 반복 훈련해 수학·코딩 벤치마크 성능이 뛰어남  \n'
 '- 구글 생태계(서치, 애드, 안드로이드 등)와 연동이 쉬워 실제 서비스 적용이 빠르고 확장성이 높음  \n'
 '- TPU 기반 효율적 학습·추론으로 대규모 모델임에도 서빙 비용이 상대적으로 낮음  \n'
 '\n'
 '비슷한 AI 모델: GPT-4, Claude-3, Llama-3, Mistral, PaLM-2, GPT-4오, 클로드-3, 라마-3, '
 '미스트랄, 팜-2')


#### PromptTemplate 의 파라미터를 배열 형태로 하여 여러개 사용하는 경우

In [6]:
# List Comprehension 예제
# before (적용하지 않은 경우)
result_list = []
for val in range(10):
    # 2의
    if val % 2 == 0:
        result_list.append(val ** 2)

result_list        

[0, 4, 16, 36, 64]

In [7]:
# after (적용한 경우)
result_list2 = [val**2 for val in range(10) if val % 2 == 0]
result_list2

[0, 4, 16, 36, 64]

In [9]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

questions = [
    {"model_name": "GPT-4", "count": 3}, #dict
    {"model_name": "Gemini", "count": 4},
    {"model_name": "Claude", "count": 4},
]

# 여러 개의 프롬프트를 미리 생성
# q변수가 dict 타입
formatted_prompts = [prompt_template.format(**q) for q in questions]
print(formatted_prompts)  # 미리 생성된 질문 목록 확인

#llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

for prompt in formatted_prompts:
    print(type(prompt), prompt)
    response = llm.invoke(prompt)
    pprint(response.content)
    print("-" * 50)

['GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.', 'Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.', 'Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.']
<class 'str'> GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.
('GPT-4는 인공신경망 기반의 트랜스포머 구조를 활용해, 방대한 텍스트 데이터에서 다음 단어를 예측하는 방식으로 사전 학습됩니다.  \n'
 '이후 인간의 피드백을 반영한 강화학습(RLHF)으로 가치 정렬을 조정해, 사용자의 질문에 도움되고 안전한 답변을 생성하도록 미세 '
 '조정됩니다.  \n'
 '결국 통계적 패턴을 학습하는 것이지, 진정한 이해·의식·지식은 없으며, 입력 프롬프트에 따라 확률적으로 적절한 문장을 만들어낼 뿐입니다.')
--------------------------------------------------
<class 'str'> Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Gemini는 텍스트, 이미지, 오디오 등 다양한 데이터를 동시에 처리하는 멀티모달 구조로 학습됩니다.  \n'
 '대규모 트랜스포머 기반 모델을 사용해 다음 토큰을 예측하는 방식으로 학습합니다.  \n'
 '지도학습과 강화학습을 결합해 정답률을 높이고, 인간의 피드백(RLHF)으로 성능을 추가로 개선합니다.  \n'
 '최종적으로는 다양한 작업에 일반화될 수 있도록 대규모 사전 학습과 미세 조정이 병행됩니다.')
--------------------------------------------------
<class 'str'> Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Claude는 대규모 텍스트 코퍼스를 미리 학습해 언어 패턴을 학습합니다.  \n'
 '사람 트레이너가 대화 흐름에서 선호되는 응답을 직접 표시하여 강화학습을 보강합니다.  \n'
 '이후 Constitutional AI 단

##### 2) ChatPromptTemplate
* Tuple 형태의 system, user, assistant 메시지 지원
* 여러 개의 메시지를 조합하여 LLM에게 전달 가능
* 간결성과 가독성이 높고 단순한 구조

In [10]:
# 2-튜플 형태의 메시지 목록으로 프롬프트 생성 (type, content)

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    # role, message
    ("system", "This system is an expert in answering questions about {topic}. \
     Please provide clear and detailed explanations."),
    ("user", "{model_name} 모델의 학습 원리를 설명해 주세요."),
])

messages = chat_prompt.format_messages(topic="AI", model_name="ExaOne")
print(messages)

# 생성한 메시지를 바로 주입하여 호출하기
#llm = ChatOpenAI(model="gpt-3.5-turbo-0125")
print(llm.model_name)
response = llm.invoke(messages)

print(type(response))
print(response.content)

[SystemMessage(content='This system is an expert in answering questions about AI.      Please provide clear and detailed explanations.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ExaOne 모델의 학습 원리를 설명해 주세요.', additional_kwargs={}, response_metadata={})]
moonshotai/kimi-k2-instruct-0905
<class 'langchain_core.messages.ai.AIMessage'>
LG AI Research에서 공개한 3,000억(30B) 파라미터 규모의 ExaOne(이하 ‘엑사원’)은 한국어/영어 얙대 언어를 동시에 다루는 생성 전용(generative) Decoder-only 트랜스포머 모델입니다.  
“학습 원리를 설명해 달라”는 질문은 기술적으로는 두 가지로 나눠 답할 수 있습니다.

A. “어떤 목표 함수(objective)로 학습이 이루어지는가?”  
B. “해당 목표 함수를 효율적으로 최적화하기 위해 어떤 인프라·기법이 쓰였는가?”

아래에 두 측면을 한꺼번에 정리했습니다. (※ 공개 논문이 아직 없으므로, LG AI Research가 발표한 블로그·발표자료·튜토리얼, 그리고 일반적으로 대규모 언어 모델(LLM) 학습에 쓰이는 기법을 조합해 설명합니다.)

--------------------------------------------------
1. 학습 목표(Objective) – “다음 토큰 예측”
--------------------------------------------------
엑사원은 표준 autoregressive language modeling을 목표로 합니다. 즉, 입력 시퀀스 x = (x₁, x₂, …, x_T)에 대해

    L = – Σ_t log P(x_t | x₁

In [11]:
# 체인을 생성하여 호출하기
#llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

chain = chat_prompt | llm | StrOutputParser()

response = chain.invoke({"topic":"AI", "model_name":"ExaOne3.5"})
print(type(response))
print(response)

<class 'str'>
LG AI Research의 ExaOne 3.5(이하 “엑사원”)는 “Large-Scale Language Model”이라는 범주에 속하지만, 1) 학습 목표, 2) 데이터 구성, 3) 학습 전략이 국내외 일반적인 LLM과는 조금 다르게 설계되어 있다. 핵심은 “한국어·과학·의료·법률·공학 등 전문 도메인에서 최고 수준의 Few-shot/Zero-shot 성능을 뽑아내되, 1조 개에 육박하는 파라미터로 인해 발생할 수 있는 ‘학습 불안정성’과 ‘추론 비용’을 최소화하는 것”이다. 이를 위해 3단계(Pre-training → Domain-Adaptive Pre-training → Fine-tuning) 파이프라인과 “Expert Mixture(Mixture-of-Experts, MoE)” 구조를 기반으로 한 희소 학습(Sparse Upcycling) 전략을 사용한다.

아래에 단계별로 원리를 정리한다.

----------------------------------------
1. Pre-training: “한국어+과학” 중심的大规模Corpus
----------------------------------------
1) 데이터 규모  
   - 약 1.5 T 토큰(한글 기준 2.5조 자)  
   - 한국어 55 %, 영어 35 %, 기타 10 %  
   - 특징: 의학 논문(PubMed/KoreaMed), 특허(KIPRIS), 한국 법률/판례, 과학기술논문(KCI, arXiv), 기술매뉴얼, 포털 뉴스, 위키, 교과서 등 “정보밀도”가 높은 문서 비중 ↑

2) 목표 함수  
   - 다음 토큰 예측(Autoregressive LM) + Sentence Order Prediction(SOP) 보조 loss  
   - 4,096 → 8,192 → 16,384 토큰으로 길이 증가 단계적 학습( curriculum learning) → 긴 context 안정성 확보

3) MoE Sparse Upcycling  
   - 밀도 모델(예

#### 3) ChatPromptTemplate
* SystemMessagePromptTemplate와 HumanMessagePromptTemplate 클래스 사용
* 객체 지향적 접근 - Message 객체를 독립적으로 생성 가능
* 여러 조건에 따라 다른 시스템 메시지 선택

```python
if user_is_beginner:
    system_message = SystemMessagePromptTemplate.from_template("초보자를 위한 설명: {topic}")
else:
    system_message = SystemMessagePromptTemplate.from_template("전문가를 위한 상세 분석: {topic}")
```

In [12]:
# ChatMessagePromptTemplate 활용

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    ChatMessagePromptTemplate
)
from langchain_openai import ChatOpenAI

# 개별 메시지 템플릿 정의
system_message = SystemMessagePromptTemplate.from_template(
    "You are an AI expert in {topic}. Please provide clear and detailed explanations."
)
user_message = HumanMessagePromptTemplate.from_template(
    "{question}"
)
ai_message = AIMessagePromptTemplate.from_template(
    "This is an example answer about {topic}."
)

# ChatPromptTemplate로 메시지들을 묶기
chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    user_message,
    ai_message
])

# 메시지 생성
messages = chat_prompt.format_messages(topic="건강", question="독감예방 접종을 꼭 맞아야 하나요?")

# LLM 호출
#llm = ChatOpenAI(model="gpt-3.5-turbo-0125")
response = llm.invoke(messages)

# 결과 출력
print(response.content)

 I will explain in Korean.

독감(인플루엔자) 예방접종은 “꼭” 맞아야 하는 것은 아니지만, “맞는 것이 훨씬 이롭다”는 것이 세계적인 의학계 공론입니다. 아래에 누가, 왜, 언제, 어떤 유형의 백신을 맞아야 하는지를 질병관리청·WHO·미국·유럽 가이드라인을 바탕으로 정리해 드릴게요.

---

### 1. 왜 독감 예방접종을 권장할까?

1. 인플루엔자는 단순 감기가 아닙니다.  
   - 고열(38 ℃ 이상), 근육통, 기침, 두통이 급격히 나타나며 1~2주간 진행됩니다.  
   - 폐렴, 뇌염, 심근염, 기존 만성질환 급성악화로 사망에 이를 수 있습니다.  
   - 대유행(팬데믹) 때는 젊은 성인도 다수 사망(2009년 신종플루 국내 사망자 226명).

2. 집단면역 효과  
   - 백신 접종률 70 % 이상이면 지역사회 전파 차단 효과가 뚜렷해, 접종 못한 고위험군(신생아, 면역저하자 등)도 보호됩니다.

3. 사회·경제적 비용 절감  
   - 국내 연구에 따르면 65세 이상에서 접종 1회당 6~8만 원의 의료비 절감, 1인당 0.8일의 생산성 손실 회수.

---

### 2. 누가 ‘우선’ 맞아야 하나?

질병청이 정한 ‘무료’ 대상이자 필수 권장군(2024년 기준):

- 만 6개월~18세 어린이·청소년  
- 임신부(모든 임신부)  
- 만 65세 이상  
- 요양시설·정신요양시설·장애인거주시설 입소자 및 종사자  
- 만성질환자(심혈관·폐질환·당뇨·신장·암·비만 등)  
- 의료인·보건인·사회복지사 등 감염 노출 직업군  
- 가금·돼지 사육 종사자, 군인, 경찰, 소방관, 교정시설 종사자

위 그룹은 인플루엔자에 걸렸을 때 입원·사망률이 3~12배 높습니다.

---

### 3. 완전히 건강한 성인도?

- 19~64세 건강 성인은 ‘국가무료’는 아니지만, WHO·CDC는 “모든 6개월 이상은 매년 접종” 권장.  
- 접종 시  
  – 발병률 40~60 % 감소(계절별 변이·일치율에 따라 다름)  
  – 

#### ChatMessagePromptTemplate는 여러 종류의 메시지(시스템, 인간, AI)를 조합하여 복잡한 프롬프트를 생성할 때 유용합니다.
* SystemMessagePromptTemplate: 이 템플릿은 AI 모델에게 역할을 부여하거나 전반적인 규칙을 설정하는 시스템 메시지를 만듭니다. 위의 예시에서는 "번역을 도와주는 유용한 도우미"라는 역할을 지정합니다.
* HumanMessagePromptTemplate: 이 템플릿은 사용자의 질문이나 요청을 담는 인간 메시지를 만듭니다. 아래의 예시에서는 번역할 텍스트를 입력받습니다.
* ChatPromptTemplate.from_messages: 이 클래스 메서드는 시스템 메시지, 인간 메시지 등 여러 종류의 MessagePromptTemplate 객체들을 리스트로 받아 하나의 채팅 프롬프트 템플릿으로 통합합니다.
* format_messages: 이 메서드는 정의된 템플릿에 실제 값을 채워 넣어 [SystemMessage, HumanMessage] 형태의 리스트를 반환합니다. 이 리스트는 채팅 모델(Chat Model) 에 바로 전달될 수 있습니다.

In [14]:
# 필요한 라이브러리 임포트
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 1. SystemMessagePromptTemplate와 HumanMessagePromptTemplate 생성
# SystemMessagePromptTemplate는 모델의 페르소나 또는 기본 지침을 설정합니다.
system_template = "You are a helpful assistant that translates {input_language} to {output_language}."
system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

# HumanMessagePromptTemplate는 사용자로부터 받는 입력 프롬프트를 정의합니다.
human_template = "{text_to_translate}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# 2. ChatPromptTemplate 생성
# 위에서 만든 두 템플릿을 리스트로 묶어 ChatPromptTemplate을 만듭니다.
chat_prompt_template = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# 3. 프롬프트 포맷팅
# chat_prompt_template.format_messages()를 사용하여 최종 메시지 리스트를 생성합니다.
# 이 함수는 딕셔너리 형태의 입력 변수를 받습니다.
formatted_prompt = chat_prompt_template.format_messages(
    input_language="English",
    output_language="Korean",
    text_to_translate="I love programming."
)

# 4. 결과 출력
print(formatted_prompt)

# LLM 호출
response = llm.invoke(formatted_prompt)

# 결과 출력
print(response.content)

[SystemMessage(content='You are a helpful assistant that translates English to Korean.', additional_kwargs={}, response_metadata={}), HumanMessage(content='I love programming.', additional_kwargs={}, response_metadata={})]
나는 프로그래밍을 사랑해.


##### 4) FewShotPromptTemplate
* FewShotPromptTemplate은 모델이 특정 형식을 따르게 하거나, 일관된 응답을 생성하도록 유도할 때 유용합니다.
* 도메인 지식이 필요하거나, AI가 오답을 줄이고 더 신뢰할 만한 답변을 생성하도록 해야 할 때 효과적입니다.

##### 4-1) PromptTemplate을 사용하지 않는 경우

In [16]:
# PromptTemplate을 사용하지 않는 경우
from langchain_openai import ChatOpenAI

# model
#llm = ChatOpenAI(model="gpt-3.5-turbo")
print(llm.model_name)

# chain 실행
result = llm.invoke("태양계의 행성들을 간략히 정리해 주세요.")

print(type(result))
print(result.content)

moonshotai/kimi-k2-instruct-0905
<class 'langchain_core.messages.ai.AIMessage'>
태양계 행성을 태양에서부터 간단히 정리하면 다음과 같습니다.

1. **수성**  
   - 가장 작고, 가장 가까운 행성  
   - 표면 온도 극단적 (낮 430°C / 밤 –180°C)

2. **금성**  
   - 두 번째 행성, ‘지구의 쌍둥이’라 불릴 만큼 크기 유사  
   - 두꺼운 CO₂ 대기 → 온실효과로 460°C에 달함

3. **지구**  
   - 유일하게 생명체가 확인된 행성  
   - 액체 물이 넓게 존재

4. **화성**  
   - ‘붉은 행성’, 얇은 대기, 고도 0 km에서도 영하 60°C  
   - 과거 물의 흔적 풍부, 탐사 로봇 상주 중

5. **목성**  
   - 태양계 최대 행성, 질량은 다른 행성 합계의 2배  
   - 대기폭풍(붉은 반점)과 79개 위성(카리스토, 유로파 등)

6. **토성**  
   - 눈에 띄는 고리(고리계)  
   - 밀도가 물보다 낮(0.7 g/cm³)

7. **천왕성**  
   - 자전축이 98° 기울어 ‘누워서’ 회전  
   - 핵심 성분이 물·암모니아·메탄 ‘아이스’

8. **해왕성**  
   - 태양계에서 가장 강한 바람(초속 600 m)  
   - 위성 ‘트리톤’은 역(逆)궤도로 공전

※ **명왕성**은 2006년 국제천문연맹(IAU) 기준으로 ‘왜행성’으로 분류됨.


##### 4-2) FewShotChatMessagePromptTemplate 사용하는 경우

In [17]:
# FewShotChatMessagePromptTemplate 사용하는 경우
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI

examples_list = [
    {
        "input": "뉴턴의 운동 법칙을 요약해 주세요.",
        "output": """### 뉴턴의 운동 법칙
1. **관성의 법칙**: 힘이 작용하지 않으면 물체는 계속 같은 상태를 유지합니다.
2. **가속도의 법칙**: 물체에 힘이 작용하면, 힘과 질량에 따라 가속도가 결정됩니다.
3. **작용-반작용 법칙**: 모든 힘에는 크기가 같고 방향이 반대인 힘이 작용합니다."""
    },
    {
        "input": "지구의 대기 구성 요소를 알려주세요.",
        "output": """### 지구 대기의 구성
- **질소 (78%)**: 대기의 대부분을 차지합니다.
- **산소 (21%)**: 생명체가 호흡하는 데 필요합니다.
- **아르곤 (0.93%)**: 반응성이 낮은 기체입니다.
- **이산화탄소 (0.04%)**: 광합성 및 온실 효과에 중요한 역할을 합니다."""
    }
]

# 예제 프롬프트 템플릿
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# FewShotChatMessagePromptTemplate 적용
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples_list,
)

# 최종 프롬프트 구성
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# 모델 생성 및 체인 구성
#llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)
chain = final_prompt | llm
print(llm.model_name)
print(chain)

# 테스트 실행
result = chain.invoke({"input": "전기의 발생 원리에 대하여 설명해 줘"})
print(result.content)

moonshotai/kimi-k2-instruct-0905
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': '뉴턴의 운동 법칙을 요약해 주세요.', 'output': '### 뉴턴의 운동 법칙\n1. **관성의 법칙**: 힘이 작용하지 않으면 물체는 계속 같은 상태를 유지합니다.\n2. **가속도의 법칙**: 물체에 힘이 작용하면, 힘과 질량에 따라 가속도가 결정됩니다.\n3. **작용-반작용 법칙**: 모든 힘에는 크기가 같고 방향이 반대인 힘이 작용합니다.'}, {'input': '지구의 대기 구성 요소를 알려주세요.', 'output': '### 지구 대기의 구성\n- **질소 (78%)**: 대기의 대부분을 차지합니다.\n- **산소 (21%)**: 생명체가 호흡하는 데 필요합니다.\n- **아르곤 (0.93%)**: 반응성이 낮은 기체입니다.\n- **이산화탄소 (0.04%)**: 광합성 및 온실 효과에 중요한 역할을 합니다.'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt

#### 5-1) PartialPrompt 
* 프롬프트를 더 동적으로 활용할 수 있으며, AI 응답을 더 일관성 있게 조정 가능함

In [18]:
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# 프롬프트 템플릿 정의 (부분 변수 적용)
prompt = PromptTemplate(
    template="{season}에 일어나는 대표적인 지구과학 현상은 {phenomenon}이 맞나요? \
        {season}에 주로 발생하는 지구과학 현상을 3개 알려주세요",
    input_variables=["phenomenon"],  # 사용자 입력 필요
    partial_variables={"season": get_current_season()}  # 동적으로 계절 값 할당
)

# OpenAI 모델 초기화
#llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

# 특정 계절의 현상 질의
query = prompt.format(phenomenon="태풍 발생")
result = llm.invoke(query)


# 결과 출력
print(f" 프롬프트: {query}")
print(f" 모델 응답: {result.content}")

 프롬프트: 겨울에 일어나는 대표적인 지구과학 현상은 태풍 발생이 맞나요?         겨울에 주로 발생하는 지구과학 현상을 3개 알려주세요
 모델 응답: 겨울철에는 태풍이 아니라 **눈/빙하 형성·북극 한파·극야 현상(극야·극주야)**이 대표적인 지구과학 현상입니다.

1. 눈·빙하 형성  
   - 영하의 기온으로 수증기가 얼음 결정로 성장 → 눈송이가 돼 눈이 내리고, 오랜 기간 쌓이면 빙하·적설층이 됩니다.

2. 북극 한파(북극진동 약화)  
   - 북극 주변 제트기류가 약해지면 극의 찬 공기가 중위도로 내려와 한파를 몰고 옵니다. 한국·미국·유럽 등에 겨울 한파가 강화되는 주요 원인입니다.

3. 극야·극주야 현상  
   - 북극·남극 지역에서 태양이 하루 혹은 몇 달 동안 지평선 아래로 들어가 계속 밤(극야)이 이어지고, 기온이 크게 떨어집니다.


In [23]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# Step 1: 현재 계절 결정
season_name = get_current_season()
#season_name = get_current_season("south")  # 계절 값 얻기
print(f"현재 계절: {season_name}")

현재 계절: 겨울


In [22]:

# Step 2: 해당 계절의 자연 현상 추천
prompt2 = ChatPromptTemplate.from_template(
    "{season}에 주로 발생하는 대표적인 지구과학 현상 3가지를 알려주세요. "
    "각 현상에 대해 간단한 설명을 포함해주세요."
)

# OpenAI 모델 사용
#llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)
# llm = ChatOpenAI(
#     #api_key=OPENAI_API_KEY,
#     base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
#     model="meta-llama/llama-4-scout-17b-16e-instruct",  # Spring AI와 동일한 모델
#     temperature=0.0
# )

print(llm.model_name)
# 체인 2: 자연 현상 추천 (입력: 계절 → 출력: 자연 현상 목록)
chain2 = (
    {"season": lambda x : season_name}  # chain1의 출력을 season 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: 현재 계절에 따른 자연 현상 추천
response = chain2.invoke({})
print(f"\n {season_name}에 발생하는 자연 현상:\n{response}")

moonshotai/kimi-k2-instruct-0905

 여름에 발생하는 자연 현상:
여름철에 빈번히 나타나는 대표적인 지구과학 현상 3가지는 다음과 같습니다.

1. 열대야  
   밤사이에도 기온이 25 ℃ 이상 떨어지지 않아 잠을 설치게 만드는 현상. 상층의 대기가 구름‧수증기로 덮여 지표의 열이 우주로 빠져나가지 못하면 생깁니다.

2. 장마(장마전선)  
   북태평양 고기압과 대륙 고기압 사이에서 형성된 정체전선이 한반도에 머물면서 연속적인 비를 뿌리는 시기. 보통 6월 말~7월 중순에 나타나며, 집중호우로 홍수를 유발할 수 있습니다.

3. 열대성 저기압(태풍)  
   북태평양에서 수온 26 ℃ 이상의 따뜻한 바닷물 위에 형성된 거대한 회전 저기압. 여름~초가을에 한반도로 북상하여 강풍과 폭우를 동반하며, 때로는 재난 수준의 피해를 줍니다.


#### 5-2) PartialPrompt 
* API 호출 데이터, 시간 정보, 사용자 정보 등을 반영할 때 매우 유용함

In [24]:
import requests
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 실시간 환율을 가져오는 함수
def get_exchange_rate():
    response = requests.get("https://api.exchangerate-api.com/v4/latest/USD")
    data = response.json()
    return f"1달러 = {data['rates']['KRW']}원"

# Partial Prompt 활용
prompt = PromptTemplate(
    template="현재 {info} 기준으로 환율 정보를 알려드립니다. 이에 대한 분석을 제공해 주세요.",
    input_variables=[],  # 사용자 입력 없음
    partial_variables={"info": get_exchange_rate()}  # API에서 가져온 데이터 자동 반영
)

# LLM 모델 설정
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# 모델에 프롬프트 전달 및 응답 받기
response = llm.invoke(prompt.format())

# 결과 출력
print(" 프롬프트:", prompt.format())
print(" 모델 응답:", response.content)

 프롬프트: 현재 1달러 = 1473.08원 기준으로 환율 정보를 알려드립니다. 이에 대한 분석을 제공해 주세요.
 모델 응답: 현재 **1달러 = 1,473.08원**은 **역대 최고 수준**의 환율로, **심리적 저항선 1,500원**을 눈앞에 둔 상황입니다. 이 환율은 단순한 숫자를 넘어 **경제 전반의 스트레스 신호**를 내포하고 있습니다. 아래에 핵심 요인과 시사점을 정리해 드립니다.

---

### 🔍 1. 환율 급등의 핵심 배경
| 요인 | 설명 |
|------|------|
| **美 금리 고정적** | 연준의 ‘higher for longer’ 기조로 미국 국채 10년물 수익률이 4.5%대를 유지 → 달러 자산 선호 |
| **韓·中 경기 둔화** | 한국 수출 6개월 연속 마이너스, 중국 경기 침체 → 원화 수요 감소 |
| **안전자산 선호** | 이스라엘-이란, 우크라이나 전쟁 지속 + 대만 긴장 → 달러·엔·스위스프랑 강세 |
| **국내 외화유동성 악화** | 4월 외환보유액이 9개월 만에 최대폭 감소(-59억 달러) → 시장에서 달러 물량 부족 체감 |
| **원화 약세 자기증식** | 수입업체·해외용 차환상환용 달러 사들이기 → 환율 상승 → 수입물가 ↑ → 인플레 악순환 |

---

### 📊 2. 민감지표 비교 (2021 vs 2024)
| 지표 | 2021년 평균 | 2024년 4월 | 변화 |
|---|---|---|---|
| 환율 | 1,144원 | 1,473원 | +29% |
| 원유(두바이) | 68$/배럴 | 89$/배럴 | +31% |
| 냉연강판 | 110만원/톤 | 140만원/톤 | +27% |
| 소비자물가 | 2.5% | 3.2% | +0.7%p |

> **결론**: 환율 29%↑가 수입물가를 떠받치며 **물가 기저효과를 상쇄** → 한은의 금리 인하 여지 축소

---

### ⚠️ 3. 단기 리스크 시나리오
- **1,500원 돌파 시**  
  – 수입업체 추가 헤지 물량 쏟아짐 → 환율 급등 